In [ ]:
!pip install unsloth
!pip install --upgrade trl transformers datasets

- 모델 로딩

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,     # 입력 시퀀스 최대 길이  더 긴 대화가 필요하면 4096으로 늘릴 수 있지만 VRAM이 소모
    load_in_4bit=True,       # 4-bit 양자화로 VRAM 절약 QLoRA의 핵심, 모델 가중치를 4-bit로 압축해서 VRAM 사용량을 크게 줄임
    full_finetuning=False,
)

- LoRA 어댑터 설정

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # 텍스트만 학습 Gemma 4는 멀티모달 모델이지만 이번엔 텍스트만 학습할 거라 비전 레이어는 제외
    finetune_language_layers=True, # 언어 생성 품질에 직접 영향을 주는 레이어라 학습 대상에 포함
    finetune_attention_modules=True, # 문맥 파악을 담당하는 어텐션 모듈을 학습
    finetune_mlp_modules=True, # 지식과 표현력이 저장된 MLP 모듈을 학습
    r=8,              # LoRA rank LoRA rank — 학습할 어댑터의 "용량". 작으면 가볍고 빠른 대신 표현력이 제한되며, 단순 스타일 학습엔 8로 충분하고, 복잡한 태스크엔 16이나 32
    lora_alpha=8, # LoRA 출력 스케일링 팩터, 보통 r과 같거나 2배로 두면 안정적
    lora_dropout=0, # 과적합 방지용 드롭아웃, 3,000개 소량 데이터에는 0으로 둬도 충분하고, Unsloth 최적화 시에도 0을 권장
    bias="none", # 바이어스 파라미터는 학습 대상에서 제외해 VRAM과 학습 파라미터 수를 줄임
    random_state=3407, # 재현성을 위한 시드 값, 숫자 자체는 임의지만 고정해두면 다음에 돌려도 같은 결과값이 나옴

    # 더 높은 Task 더 복잡한 태스크(예: 코드 생성, 전문 도메인 지식)를 학습시키려면 r을 16 또는 32로 올리고, lora_alpha도 같이 올려주면 됨
)

-  데이터셋 준비

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

dataset = load_dataset("mlabonne/FineTome-100k", split="train[:3000]") # FineTome-100k은 고품질 instruction-following 데이터셋 실전에서는 전체 100k 또는 자체 데이터를 사용, 데이터가 많아지면 학습 시간과 비용도 비례해서 증가됨
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)1

- 내 데이터로 학습하고 싶다면, 같은 conversations 포맷으로 JSON 파일을 만들고 적용

[
  {
    "conversations": [
      {"from": "human", "value": "질문"},
      {"from": "gpt", "value": "답변"}
    ]
  }
]

- 학습 시작

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only # train_on_responses_only는 모델이 답변 부분만 학습하도록 설정함 질문까지 학습하면 모델이 질문을 따라하는 패턴이 생김

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        output_dir="/shared/gemma4-finetuned", # 체크포인트는 Object Storage(/shared)에 저장됨 학습이 중단돼도 이어서 할 수 있음
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

trainer_stats = trainer.train()

- 추론 테스트

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": [
        {"type": "text", "text": "Explain quantum computing in simple terms."}
    ]}
]

_ = model.generate(
    **tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda"), # 자체 내장 cpu 사용할거면 cpu, 그래픽카드 사용은 cuda
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7, # 가중치 0.7로 설정
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

- 모델 저장
Object Storage에 저장 후

워크스페이스를 Stop해도 모델이 유지됨
같은 조직의 다른 워크스페이스에서 바로 로딩할 수 있어요
팀원이 같은 Object Storage를 마운트하면 모델을 바로 사용할 수 있음

In [ ]:
model.save_pretrained("/shared/gemma4-finetuned/final")
tokenizer.save_pretrained("/shared/gemma4-finetuned/final")

### 해당 모델 학습 시킨 후 적용 가능한 도메인
- 내 데이터로 학습: 고객 FAQ, 제품 문서, 도메인 지식으로 전문가 모델 만들기
- DPO/ORPO: 답변 선호도 학습으로 더 자연스러운 응답 만들기
- 더 큰 모델: Gemma 4 27B나 Llama 4 Scout로 스케일업
- GGUF 변환: Ollama나 llama.cpp에서 로컬 추론용으로 변환

- 대화형 챗봇 및 가상 비서: 사용자 질의에 대해 대화 형식으로 응답하는 챗봇에 GPT가 핵심 엔진으로 사용된다. 예를 들어 ChatGPT는 일반적인 질의응답부터 상담, 창작 대화까지 수행하는 만능 챗봇이며, 각종 웹사이트의 고객지원 챗봇이나 음성 비서에도 GPT 모델이 탑재되어 더 자연스러운 대화와 맞춤형 응대를 제공한다.
- 콘텐츠 생성 및 글쓰기 보조: GPT의 텍스트 생성 능력을 활용해 블로그 글, 기사, 마케팅 카피, 소셜미디어 게시물, 시나리오 등 다양한 글을 자동 작성하거나 초안을 생성하는 데 활용한다. 예를 들어 소설 줄거리 생성, 유튜브 영상 설명 작성, 이메일 초안 만들기 등에 GPT가 사용된다. 작가나 마케팅 분야에선 GPT를 아이디어 발상 및 글쓰기 도우미로 활용하고 있으며, Microsoft Word 등 문서 편집기에도 GPT 기반 작성 보조 기능이 통합되고 있다. (단, 생성된 콘텐츠의 진실성 및 저작권 이슈에 주의하여 전문가의 감수를 거치는 것이 권장된다.)
- 번역 및 다국어 처리: GPT는 다수의 언어로 학습되었기 때문에 영어 ↔ 한국어를 비롯한 자연스러운 기계 번역에도 활용될 수 있다. 특히 GPT-4는 맥락을 고려한 고품질 번역을 수행할 수 있고, 음성 인식/합성 기술과 결합하여 실시간 통역과 같은 응용도 가능성을 보이고 있다. 한편 GPT의 다국어 이해 능력은 번역 외에도 외국어 학습 보조(Q\&A 형식의 설명 등)에도 응용되고 있다.
- 문서 요약 및 텍스트 정리: 긴 문서를 읽고 핵심을 뽑아 요약문을 생성하는 작업에 GPT가 두각을 나타낸다. 예컨대 수십 페이지의 보고서를 GPT에게 입력하면 몇 문장으로 개요를 요약해주거나, 회의 녹취록을 읽고 액션 아이템을 정리해주는 등 지식 노동 자동화에 활용된다. 또한 사용자의 지시에 따라 특정 스타일로 다시 쓰기(예: “간결한 문체로 바꿔줘”)와 같은 텍스트 변환 작업도 가능하다.
- 데이터 분석 및 시각화: 자연어로 된 데이터(설문 응답, 고객 리뷰 등)를 GPT가 분석하여 패턴이나 통계를 추출하거나 인사이트를 제공하는 실험이 진행되고 있다. 예를 들어 “지난 달 고객 피드백에서 주요 불만 사항이 무엇인지 요약해줘”라는 질의에 GPT는 다수의 텍스트를 분석하여 요점을 추려준다. 또 GPT의 출력 결과를 다른 도구와 연계해 차트나 그래프로 시각화하는 응용도 가능하다. 다만 기업 내부 데이터 등을 다룰 때는 프라이버시와 보안 문제가 있을 수 있어 데이터 입력에 신중을 기하고 있다.
- 프로그래밍 보조 및 코드 생성: GPT는 자연어 뿐 아니라 주요 프로그래밍 언어에 대해서도 학습되어, 코드 자동생성에 활용된다. OpenAI가 GPT-3를 파인튜닝해 공개한 Codex 모델은 파이썬 등 코드 완성 능력이 뛰어나며, 이를 기반으로 한 GitHub Copilot은 개발자가 주석으로 의도를 쓰면 대응되는 코드를 실시간으로 제안해주는 도구로 각광받고 있다. GPT-4는 더욱 발전된 코딩 능력을 보여주어, 간단한 함수부터 인터넷 검색 기능을 갖춘 복잡한 코드까지 생성 가능하며, 디버깅이나 코드 리뷰에도 활용되고 있다. 다만 생성된 코드의 정확도는 검증이 필요하며, GPT가 생성한 코드에도 훈련 데이터의 라이선스 문제가 있을 수 있어 주의 깊은 검토가 요구된다.
- 교육 및 의료 분야: GPT는 교육 영역에서 개인교사나 학습 파트너로 활용 가능성이 주목된다. 학생들이 어려운 개념을 질문하면 GPT가 단계별로 설명하거나 추가 예제를 제공하는 등, 상시 이용 가능한 AI 튜터 역할을 할 수 있다. 실제로 Khan Academy 등에서 GPT-4 기반 학습 도우미를 시험 중이며, 초기 결과는 긍정적이다. 의료 분야에서도 GPT의 응용이 활발한데, 예를 들어 초기 문진에 응답하여 증상에 맞는 조언을 제공하거나, 방대한 의학 논문을 요약 정리해 의사 결정을 돕는 등의 활용이 연구되고 있다. 특히 의료 정보 접근성이 낮은 지역에서 GPT 기반 챗봇이 환자 상담에 활용될 경우 원격 의료에 도움을 줄 수 있다는 전망도 있다. 다만 교육과 의료 모두 인간 전문가의 역할을 대체하기보다는 보조적 수단으로 활용하며, 잘못된 정보 제공을 막기 위한 검증 단계가 필요하다.